# Mental Health in Tech — Exploratory Data Analysis

This notebook explores the **OSMI Mental Health in Tech Survey** dataset (`survey.csv`): 1,259 responses from tech-industry workers about mental health attitudes, workplace support, and treatment-seeking behavior.

**Contents**
1. Setup & Data Loading
2. Data Overview & Missing Values
3. Data Cleaning (Age, Gender)
4. Univariate Analysis (distributions)
5. Bivariate Analysis (relationships with `treatment`)
6. Correlation Analysis
7. Key Takeaways


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.titleweight"] = "bold"

pd.set_option("display.max_columns", 100)


## 1. Load the Data

In [ ]:
df = pd.read_csv("data/survey.csv")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

## 2. Missing Values

`comments` and `state` have the most missing values (free text / US-only field). `work_interfere` and `self_employed` also have gaps.

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(missing.index[::-1], missing_pct.values[::-1], color="#e07a5f")
ax.set_xlabel("% Missing")
ax.set_title("Missing Values by Column")
for bar, val in zip(bars, missing_pct.values[::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f"{val}%", va="center")
plt.tight_layout()
plt.show()

## 3. Data Cleaning

Two columns need cleanup before analysis:
- **Age**: contains clearly invalid values (negative numbers, values in the billions) — a data entry/bot artifact common in this public dataset. We clip to a plausible working-age range (18–75).
- **Gender**: free-text field with ~49 distinct spellings of essentially 3 categories (Male / Female / Other). We normalize it.

In [ ]:
print("Age range before cleaning:", df['Age'].min(), "to", df['Age'].max())

df_clean = df.copy()
df_clean.loc[(df_clean['Age'] < 18) | (df_clean['Age'] > 75), 'Age'] = np.nan

n_dropped = df_clean['Age'].isna().sum() - df['Age'].isna().sum()
print(f"Flagged {n_dropped} implausible Age values as NaN")
print("Age range after cleaning:", df_clean['Age'].min(), "to", df_clean['Age'].max())

In [ ]:
def normalize_gender(g):
    g = str(g).strip().lower()
    male_terms = {"male", "m", "man", "cis male", "cis man", "make", "malr", "maile",
                  "mal", "male (cis)", "male-ish", "msle", "mail", "guy (-ish) ^_^",
                  "male leaning androgynous", "ostensibly male, unsure what that really means"}
    female_terms = {"female", "f", "woman", "cis female", "cis-female/femme", "femake",
                     "femail", "female (cis)", "female (trans)", "trans woman", "trans-female"}
    if g in male_terms:
        return "Male"
    if g in female_terms:
        return "Female"
    return "Other/Non-binary"

df_clean['Gender_clean'] = df_clean['Gender'].apply(normalize_gender)
df_clean['Gender_clean'].value_counts()

## 4. Univariate Analysis

### 4.1 Age Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean['Age'].dropna(), bins=30, kde=True, color="#3d5a80", ax=axes[0])
axes[0].axvline(df_clean['Age'].median(), color="#e07a5f", linestyle="--", label=f"Median: {df_clean['Age'].median():.0f}")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")
axes[0].legend()

sns.boxplot(y=df_clean['Age'].dropna(), color="#81b29a", ax=axes[1])
axes[1].set_title("Age — Boxplot")
axes[1].set_ylabel("Age")

plt.tight_layout()
plt.show()

### 4.2 Gender Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
order = df_clean['Gender_clean'].value_counts().index
sns.countplot(y=df_clean['Gender_clean'], order=order, hue=df_clean['Gender_clean'], palette="Set2", legend=False, ax=ax)
ax.set_title("Respondents by Gender")
ax.set_xlabel("Count")
ax.set_ylabel("")
for p in ax.patches:
    ax.text(p.get_width() + 5, p.get_y() + p.get_height()/2, int(p.get_width()), va="center")
plt.tight_layout()
plt.show()

### 4.3 Top Countries

In [ ]:
top_countries = df_clean['Country'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, hue=top_countries.index, palette="crest", legend=False, ax=ax)
ax.set_title("Top 15 Countries by Respondent Count")
ax.set_xlabel("Count")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 4.4 Company Size & Remote Work

In [ ]:
order_size = ["1-5", "6-25", "26-100", "100-500", "500-1000", "More than 1000"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(y=df_clean['no_employees'], order=order_size, hue=df_clean['no_employees'], palette="flare", legend=False, ax=axes[0])
axes[0].set_title("Company Size")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("")

remote_counts = df_clean['remote_work'].value_counts()
axes[1].pie(remote_counts.values, labels=remote_counts.index, autopct="%1.1f%%",
            colors=["#3d5a80", "#e07a5f"], startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title("Remote Work")

plt.tight_layout()
plt.show()

### 4.5 Treatment-Seeking & Family History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col, title in zip(
    axes,
    ["treatment", "family_history", "work_interfere"],
    ["Sought Treatment", "Family History of Mental Illness", "Work Interference"]
):
    counts = df_clean[col].value_counts()
    sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="Set2", legend=False, ax=ax)
    ax.set_title(title)
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

### 4.6 Workplace Support Indicators

In [ ]:
support_cols = ["benefits", "care_options", "wellness_program", "seek_help", "anonymity"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, col in zip(axes, support_cols):
    counts = df_clean[col].value_counts()
    sns.barplot(x=counts.index, y=counts.values, hue=counts.index, palette="viridis", legend=False, ax=ax)
    ax.set_title(col.replace("_", " ").title())
    ax.set_ylabel("Count")
    ax.tick_params(axis='x', rotation=20)

axes[-1].axis("off")
plt.tight_layout()
plt.show()

## 5. Bivariate Analysis — Relationship with `treatment`

### 5.1 Age vs Treatment

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df_clean, x="treatment", y="Age", hue="treatment", palette="Set2", legend=False, ax=ax)
sns.stripplot(data=df_clean, x="treatment", y="Age", color="black", alpha=0.15, size=2, ax=ax)
ax.set_title("Age Distribution by Treatment-Seeking")
plt.tight_layout()
plt.show()

### 5.2 Gender vs Treatment

In [ ]:
ct = pd.crosstab(df_clean['Gender_clean'], df_clean['treatment'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(8, 5))
ct.plot(kind="bar", stacked=True, color=["#e07a5f", "#3d5a80"], ax=ax)
ax.set_title("Treatment-Seeking Rate by Gender (%)")
ax.set_ylabel("% of Group")
ax.set_xlabel("")
ax.legend(title="Treatment")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 5.3 Family History vs Treatment

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(data=df_clean, x="family_history", hue="treatment", palette="Set2", ax=ax)
ax.set_title("Family History vs Treatment-Seeking")
ax.set_xlabel("Family History of Mental Illness")
ax.legend(title="Sought Treatment")
plt.tight_layout()
plt.show()

### 5.4 Work Interference vs Treatment

In [ ]:
order_wi = ["Never", "Rarely", "Sometimes", "Often"]
fig, ax = plt.subplots(figsize=(8, 5))
sns.countplot(data=df_clean, x="work_interfere", hue="treatment", order=order_wi, palette="Set2", ax=ax)
ax.set_title("Work Interference vs Treatment-Seeking")
ax.set_xlabel("How Often Mental Health Interferes with Work")
ax.legend(title="Sought Treatment")
plt.tight_layout()
plt.show()

### 5.5 Comfort Discussing Mental Health — Coworkers vs Supervisor

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(axes, ["coworkers", "supervisor"], ["Comfort with Coworkers", "Comfort with Supervisor"]):
    counts = df_clean[col].value_counts()
    ax.pie(counts.values, labels=counts.index, autopct="%1.1f%%", startangle=90,
           colors=sns.color_palette("Set2", len(counts)),
           wedgeprops={"edgecolor": "white", "linewidth": 1.5})
    ax.set_title(f"{title} re: Mental Health")

plt.tight_layout()
plt.show()

### 5.6 Mental Health vs Physical Health Consequence Perception

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
mh = df_clean['mental_health_consequence'].value_counts()
ph = df_clean['phys_health_consequence'].value_counts()

x = np.arange(len(mh.index))
width = 0.35
ax.bar(x - width/2, mh.values, width, label="Mental Health", color="#3d5a80")
ax.bar(x + width/2, [ph.get(i, 0) for i in mh.index], width, label="Physical Health", color="#e07a5f")
ax.set_xticks(x)
ax.set_xticklabels(mh.index)
ax.set_ylabel("Count")
ax.set_title("Perceived Negative Consequence: Mental vs Physical Health Discussion")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

We label-encode the key categorical columns to inspect pairwise associations.

In [ ]:
from sklearn.preprocessing import LabelEncoder

corr_cols = ["treatment", "family_history", "work_interfere", "benefits", "care_options",
             "wellness_program", "seek_help", "anonymity", "mental_health_consequence",
             "phys_health_consequence", "coworkers", "supervisor", "obs_consequence", "remote_work"]

df_enc = df_clean[corr_cols].copy()
for c in corr_cols:
    df_enc[c] = LabelEncoder().fit_transform(df_enc[c].astype(str))

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(df_enc.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Correlation Heatmap — Key Workplace & Mental Health Variables")
plt.tight_layout()
plt.show()

## 7. Key Takeaways

- Roughly half of respondents report having **sought treatment** for a mental health condition.
- **Family history** of mental illness shows a strong positive relationship with treatment-seeking — the single strongest correlate in the heatmap.
- Respondents who say mental health issues **"Often"** interfere with their work are far more likely to have sought treatment than those who say **"Never"**.
- **Comfort discussing mental health** is consistently lower than comfort discussing physical health across coworkers/supervisor/consequence questions.
- The sample **skews male** (~79% after normalization) and is dominated by the **United States** and **United Kingdom**.
- Company **benefits, wellness programs, and anonymity assurances** are inconsistently known/offered — a large share of respondents answer "Don't know," pointing to a communication gap between HR policy and employee awareness.

*Next steps: a predictive model (e.g. logistic regression / random forest) on `treatment` using these features, or a deeper text analysis of the `comments` field.*